# 实验二 B：经典机器学习算法性能对比分析

**实验目的**

1. 基于 Wine Dataset（红酒分类数据集），掌握分类任务的**数据预处理**与**特征建模**流程；
2. 实现多种典型机器学习算法（支持向量机、逻辑回归、K 近邻、决策树、随机森林、朴素贝叶斯、LDA），
   理解不同模型在特征空间中**决策边界**的差异；
3. 结合准确率、混淆矩阵、交叉验证等指标，对各类算法做**系统的性能对比分析**。

**数据集**：[`sklearn.datasets.load_wine`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html)
——178 个样本、13 个化学成分特征（酒精度、苹果酸、类黄酮……）、3 个类别
（`class_0 / class_1 / class_2`），类别比例约为 33% : 40% : 27%，属于小样本、多特征、类别基本均衡的分类问题。

In [1]:
import time
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots

from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_score,
                             recall_score)
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

pio.templates.default = "plotly_white"
SEED = 42

# 配色：三个类别用验证过色觉安全性的分类色；单序列图统一用蓝色
CLASS_COLORS = ["#2a78d6", "#eb6834", "#1baf7a"]
C_BLUE, C_ORANGE, C_RED = "#2a78d6", "#eb6834", "#e34948"
C_INK, C_MUTED, C_DATA = "#0b0b0b", "#52514e", "#8a8a86"
# 蓝色顺序色阶（浅 -> 深）：用于混淆矩阵、决策区域等“大小”类编码
BLUES = [[0.0, "#f2f7fe"], [0.35, "#cde2fb"], [0.7, "#6da7ec"], [1.0, "#2a78d6"]]
# 决策区域的浅色底：与类别色同色系，保证区域与散点一一对应
REGION_TINTS = ["#d8e7fa", "#fadcce", "#cfeee0"]


def style(fig, title="", xlabel="", ylabel="", width=760, height=430, legend_bottom=True):
    """统一的图形样式：标题、轴标签、底部图例、留白"""
    fig.update_layout(
        title=dict(text=title, x=0.02, xanchor="left"),
        xaxis_title=xlabel, yaxis_title=ylabel,
        width=width, height=height,
        margin=dict(l=70, r=30, t=70, b=80),
        legend=dict(orientation="h", yanchor="top", y=-0.18, x=0, title=None),
        hovermode="closest",
    )
    return fig


print("scikit-learn", __import__("sklearn").__version__, "| numpy", np.__version__)

scikit-learn 1.9.0 | numpy 2.4.6


## 一、数据加载与探索性分析

先把数据读成 DataFrame，检查样本规模、类别分布、缺失值与重复样本，
再比较各特征对类别的区分能力——这一步决定后面“特征建模”时该如何挑选特征。

In [2]:
wine = load_wine(as_frame=True)
df = wine.frame.copy()
df["class_name"] = df["target"].map(dict(enumerate(wine.target_names)))

X_all = df[wine.feature_names].to_numpy()
y_all = df["target"].to_numpy()

print(f"样本数 {X_all.shape[0]}，特征数 {X_all.shape[1]}")
print(f"类别标签：{[str(n) for n in wine.target_names]}")
print(f"类别分布：{ {str(k): int(v) for k, v in df['class_name'].value_counts().reindex(wine.target_names).items()} }")
print(f"缺失值总数：{int(df.isna().sum().sum())}")
print(f"重复样本数：{int(df.duplicated().sum())}")
df.head()

样本数 178，特征数 13
类别标签：['class_0', 'class_1', 'class_2']
类别分布：{'class_0': 59, 'class_1': 71, 'class_2': 48}
缺失值总数：0
重复样本数：0


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target,class_name
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0,class_0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0,class_0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0,class_0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0,class_0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0,class_0


In [3]:
# 用单因素方差分析（F 检验）衡量每个特征对类别的区分能力，F 值越大区分能力越强
f_scores, p_values = f_classif(X_all, y_all)
feat_rank = pd.DataFrame({
    "F 值": np.round(f_scores, 1),
    "p 值": [f"{p:.2e}" for p in p_values],
    "类间均值差异": [f"{df.groupby('class_name')[f].mean().max() - df.groupby('class_name')[f].mean().min():.2f}"
                 for f in wine.feature_names],
}, index=wine.feature_names).sort_values("F 值", ascending=False)

TOP_FEATURES = feat_rank.index[:4].tolist()
print("区分能力最强的 4 个特征：", TOP_FEATURES)
feat_rank

区分能力最强的 4 个特征： ['flavanoids', 'proline', 'od280/od315_of_diluted_wines', 'alcohol']


,F 值,p 值,类间均值差异
flavanoids,233.9,3.60e-50,2.20
proline,207.9,5.78e-47,596.20
od280/od315_of_diluted_wines,190.0,1.39e-44,1.47
alcohol,135.1,3.32e-36,1.47
color_intensity,120.7,1.16e-33,4.31
hue,101.3,5.92e-30,0.38
total_phenols,93.7,2.14e-28,1.16
malic_acid,36.9,4.13e-14,1.40
alcalinity_of_ash,35.8,9.44e-14,4.38
proanthocyanins,30.3,5.13e-12,0.75


In [4]:
fig = make_subplots(rows=1, cols=2, column_widths=[0.34, 0.66],
                    subplot_titles=("三个类别的样本数", "13 个特征之间的相关系数（蓝色负相关、红色正相关）"))

counts = df["class_name"].value_counts().reindex(wine.target_names)
fig.add_trace(go.Bar(
    x=counts.index, y=counts.values, marker_color=C_BLUE, showlegend=False,
    text=counts.values, textposition="outside", textfont=dict(color=C_MUTED),
    hovertemplate="%{x}<br>样本数 %{y}<extra></extra>",
), row=1, col=1)

corr = df[wine.feature_names].corr()
fig.add_trace(go.Heatmap(
    z=corr.to_numpy(), x=corr.columns, y=corr.columns,
    zmin=-1, zmax=1, colorscale=[[0.0, "#2a78d6"], [0.5, "#f0efec"], [1.0, "#e34948"]],
    colorbar=dict(title="r", thickness=13, len=0.9),
    hovertemplate="%{y} ~ %{x}<br>r = %{z:.2f}<extra></extra>",
), row=1, col=2)

fig.update_yaxes(title_text="样本数", range=[0, counts.max() * 1.2], row=1, col=1)
style(fig, "数据探索：类别分布与特征相关性", "", "", width=1120, height=520)
fig.show()

In [5]:
# 4 个最具区分度特征在每个类别下的分布
# 注意：proline 的量级是其他特征的千倍，绝不能共用一个纵轴，否则三个小量级特征会被压成一条线，
# 这里用“小倍数”（每个特征一个子图、各自独立的纵轴）来呈现。
fig = make_subplots(rows=1, cols=len(TOP_FEATURES), horizontal_spacing=0.07,
                    subplot_titles=TOP_FEATURES)

for k, feat in enumerate(TOP_FEATURES, start=1):
    for cname, color, tint in zip(wine.target_names, CLASS_COLORS, REGION_TINTS):
        fig.add_trace(go.Box(
            y=df.loc[df["class_name"] == cname, feat], name=cname,
            legendgroup=cname, showlegend=(k == 1),
            marker_color=color, line=dict(width=2), fillcolor=tint, boxpoints=False,
            hovertemplate=f"{cname}<br>%{{y:.2f}}<extra></extra>",
        ), row=1, col=k)

fig.update_yaxes(title_text="特征取值", row=1, col=1)
style(fig, "区分能力最强的 4 个特征在不同类别下的分布（每个特征独立纵轴）", "", "",
      width=1120, height=460)
fig.show()

从箱线图可以看到，`flavanoids`、`od280/od315_of_diluted_wines`、`color_intensity` 等特征在三个类别间的分布有明显错位，
单个特征就具备不错的判别力；但各特征之间也存在相关（例如 `total_phenols` 与 `flavanoids` 的相关系数接近 0.86），
因此这类问题适合用能综合多个特征的分类器，而不是逐个特征做阈值判断。

## 二、数据预处理：划分数据集与标准化

分类实验的第一步是把数据划分为**训练集**与**测试集**，并且划分时保持类别比例一致（分层抽样），
避免某个类别在测试集中过多或过少。第二步是**标准化**：13 个特征的量纲差别很大
（如 `proline` 的取值在 300–1700，而 `hue` 只有 0.5–1.7），
对 SVM、KNN、逻辑回归这类依赖距离或梯度的方法，不标准化会让大量纲特征主导模型。

标准化参数**只在训练集上估计**，再用同一组均值方差去变换测试集，以避免测试集信息泄漏。
下面的模型统一用 `make_pipeline(StandardScaler(), 模型)` 组合，保证交叉验证过程中标准化也被正确地“放进折内”。

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.3, random_state=SEED, stratify=y_all)

print(f"训练集 {X_train.shape[0]} 条，测试集 {X_test.shape[0]} 条")
print(f"训练集类别分布 {np.bincount(y_train)}，测试集类别分布 {np.bincount(y_test)}")

scaler_demo = StandardScaler().fit(X_train)
X_train_std = scaler_demo.transform(X_train)
compare = pd.DataFrame({
    "原始 mean": X_train.mean(axis=0),
    "原始 std": X_train.std(axis=0),
    "标准化后 mean": X_train_std.mean(axis=0),
    "标准化后 std": X_train_std.std(axis=0),
}, index=wine.feature_names).round(3)

print("\n量纲差异最大的几个特征（标准化前后对比）：")
compare.reindex(compare["原始 std"].sort_values(ascending=False).index).head(5)

训练集 124 条，测试集 54 条
训练集类别分布 [41 50 33]，测试集类别分布 [18 21 15]

量纲差异最大的几个特征（标准化前后对比）：


,原始 mean,原始 std,标准化后 mean,标准化后 std
proline,748.331,306.825,-0.0,1.0
magnesium,100.040,15.172,-0.0,1.0
alcalinity_of_ash,19.669,3.373,0.0,1.0
color_intensity,4.994,2.371,-0.0,1.0
malic_acid,2.287,1.056,0.0,1.0


## 三、八种算法的训练与评估

选取 8 种典型算法，覆盖四类归纳偏好：

| 类别 | 算法 | 决策边界形态 |
|---|---|---|
| 线性模型 | 逻辑回归、线性 SVM、LDA | 线性超平面 |
| 核方法 | 高斯核 SVM (RBF) | 光滑的非线性边界 |
| 实例方法 | K 近邻 (k=5) | 分片拼接的非线性边界 |
| 树模型 | 决策树、随机森林 | 轴平行的矩形划分 |
| 概率模型 | 高斯朴素贝叶斯 | 二次曲面（各特征独立假设） |

评估方式：在训练集上做 **5 折分层交叉验证**（衡量稳定性、避免测试集泄漏），
再在完全没参与训练的**测试集**上评估准确率、宏平均精确率/召回率/F1，并统计混淆矩阵。

In [7]:
models = {
    "逻辑回归": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEED)),
    "线性 SVM": make_pipeline(StandardScaler(), SVC(kernel="linear", C=1.0, random_state=SEED)),
    "高斯核 SVM (RBF)": make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale", random_state=SEED)),
    "K 近邻 (k=5)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "决策树": DecisionTreeClassifier(random_state=SEED),
    "随机森林": RandomForestClassifier(n_estimators=300, random_state=SEED),
    "高斯朴素贝叶斯": GaussianNB(),
    "线性判别分析 (LDA)": make_pipeline(StandardScaler(), LinearDiscriminantAnalysis()),
}
print(f"共 {len(models)} 个模型：", "、".join(models))

共 8 个模型： 逻辑回归、线性 SVM、高斯核 SVM (RBF)、K 近邻 (k=5)、决策树、随机森林、高斯朴素贝叶斯、线性判别分析 (LDA)


In [8]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
rows, predictions, cv_scores = [], {}, {}

for name, model in models.items():
    t0 = time.perf_counter()
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy")
    model.fit(X_train, y_train)
    fit_ms = (time.perf_counter() - t0) * 1000

    y_pred = model.predict(X_test)
    predictions[name] = y_pred
    cv_scores[name] = scores
    rows.append({
        "5 折 CV 准确率": round(scores.mean(), 4),
        "CV 标准差": round(scores.std(), 4),
        "测试集准确率": round(accuracy_score(y_test, y_pred), 4),
        "宏平均精确率": round(precision_score(y_test, y_pred, average="macro"), 4),
        "宏平均召回率": round(recall_score(y_test, y_pred, average="macro"), 4),
        "宏平均 F1": round(f1_score(y_test, y_pred, average="macro"), 4),
        "训练耗时(ms)": round(fit_ms, 1),
    })

df_scores = (pd.DataFrame(rows, index=list(models))
             .sort_values("测试集准确率", ascending=False))
df_scores

,5 折 CV 准确率,CV 标准差,测试集准确率,宏平均精确率,宏平均召回率,宏平均 F1,训练耗时(ms)
随机森林,0.9760,0.0320,1.0000,1.0000,1.0000,1.0000,822.8
高斯朴素贝叶斯,0.9600,0.0506,1.0000,1.0000,1.0000,1.0000,5.0
逻辑回归,0.9840,0.0196,0.9815,0.9825,0.9841,0.9829,19.4
高斯核 SVM (RBF),0.9920,0.0160,0.9815,0.9848,0.9778,0.9808,9.2
线性判别分析 (LDA),0.9920,0.0160,0.9815,0.9825,0.9841,0.9829,8.5
线性 SVM,0.9840,0.0196,0.9630,0.9666,0.9619,0.9636,9.2
决策树,0.8783,0.0590,0.9630,0.9710,0.9593,0.9638,6.7
K 近邻 (k=5),0.9680,0.0299,0.9444,0.9444,0.9524,0.9441,11.0


In [9]:
# 准确率对比：点图 + 交叉验证标准差（点图不需要从 0 开始的基线，便于看清差异）
order = df_scores.index.tolist()
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_scores["5 折 CV 准确率"], y=order, mode="markers", name="5 折 CV 准确率（训练集）",
    marker=dict(size=11, color=C_BLUE, line=dict(width=1.5, color="#fcfcfb")),
    error_x=dict(type="data", array=df_scores["CV 标准差"], color=C_BLUE, thickness=1.5, width=4),
    hovertemplate="%{y}<br>CV 准确率 %{x:.4f}<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=df_scores["测试集准确率"], y=order, mode="markers", name="测试集准确率（54 条）",
    marker=dict(size=11, color=C_ORANGE, symbol="diamond", line=dict(width=1.5, color="#fcfcfb")),
    hovertemplate="%{y}<br>测试集准确率 %{x:.4f}<extra></extra>",
))
fig.update_xaxes(range=[0.80, 1.02], dtick=0.025, title_text="准确率")
fig.update_yaxes(title_text="", autorange="reversed")
style(fig, "八种算法的准确率对比（横轴为准确率，误差棒为 CV 的 5 折标准差）",
      "准确率", "", width=820, height=560)
fig.show()

In [10]:
# 混淆矩阵：2 行 4 列，行 = 真实类别，列 = 预测类别
names = df_scores.index.tolist()
fig = make_subplots(rows=2, cols=4, horizontal_spacing=0.10, vertical_spacing=0.24,
                    subplot_titles=[f"{n}<br>准确率 {accuracy_score(y_test, predictions[n]):.3f}"
                                    for n in names])

labels = [str(c) for c in wine.target_names]
for k, name in enumerate(names):
    cm = confusion_matrix(y_test, predictions[name])
    fig.add_trace(go.Heatmap(
        # 反转 y 轴，让真实类别 class_0 显示在最上面，符合混淆矩阵的常规读法
        z=cm[::-1], x=[f"预测 {c}" for c in labels], y=[f"真实 {c}" for c in labels[::-1]],
        colorscale=BLUES, zmin=0, zmax=cm.max(), showscale=False,
        text=cm[::-1], texttemplate="%{text}", textfont=dict(size=13, color=C_INK),
        hovertemplate="%{y} → %{x}<br>%{z} 个<extra></extra>",
    ), row=k // 4 + 1, col=k % 4 + 1)

style(fig, "八种算法的混淆矩阵（色深表示样本数，对角线越深越好）", "", "",
      width=1180, height=730, legend_bottom=False)
fig.update_layout(margin=dict(l=70, r=30, t=115, b=60))
fig.show()

In [11]:
# 准确率最高的模型，输出逐类别的详细报告
best_name = df_scores.index[0]
print(f"测试集准确率最高的模型：{best_name}\n")
print(classification_report(y_test, predictions[best_name], target_names=wine.target_names, digits=3))

测试集准确率最高的模型：随机森林

              precision    recall  f1-score   support

     class_0      1.000     1.000     1.000        18
     class_1      1.000     1.000     1.000        21
     class_2      1.000     1.000     1.000        15

    accuracy                          1.000        54
   macro avg      1.000     1.000     1.000        54
weighted avg      1.000     1.000     1.000        54



## 四、决策边界对比

准确率只给出一个数字，看不清模型“怎么分类”。把 13 维特征**先标准化、再经 PCA 降到 2 维**，
在这张二维平面上重新训练各类模型，就可把决策边界画出来（二维投影只用于**观察边界形态**，
性能评估仍以上面 13 维的结果为准）。图中浅色区域是模型的预测区域，圆点是训练样本。

注意 PCA 之前必须先标准化：PCA 找的是方差最大的方向，若不标准化，
量纲最大的 `proline`（取值 300–1700）会独占第一主成分，降维结果只反映它一个特征。

In [12]:
# 标准化 + PCA 组成一条流水线：标准化保证各特征等权，PCA 才能找到真正的主方向
projection = make_pipeline(StandardScaler(), PCA(n_components=2, random_state=SEED))
X_pca = projection.fit_transform(X_train)
X_test_pca = projection.transform(X_test)
ratio = projection[-1].explained_variance_ratio_
print("两个主成分解释的方差比例：", np.round(ratio, 3), f"（合计 {ratio.sum():.1%}）")
print(f"投影后 PC1 范围 [{X_pca[:, 0].min():.2f}, {X_pca[:, 0].max():.2f}]，"
      f"PC2 范围 [{X_pca[:, 1].min():.2f}, {X_pca[:, 1].max():.2f}]")

# 网格分辨率按数据范围自适应，保证总点数可控（约 220×220，四种模型合计约 19 万次预测）
pad, n_grid = 0.8, 320
gx = np.linspace(X_pca[:, 0].min() - pad, X_pca[:, 0].max() + pad, n_grid)
gy = np.linspace(X_pca[:, 1].min() - pad, X_pca[:, 1].max() + pad, n_grid)
xx, yy = np.meshgrid(gx, gy)
grid = np.c_[xx.ravel(), yy.ravel()]
print(f"决策平面网格：{n_grid} × {n_grid} = {grid.shape[0]:,} 个点")

# 决策区域用 3 段“硬边界”色阶：0、1、2 三个类别各占一段浅色
region_scale = [[0.0, REGION_TINTS[0]], [0.32, REGION_TINTS[0]],
                [0.34, REGION_TINTS[1]], [0.66, REGION_TINTS[1]],
                [0.68, REGION_TINTS[2]], [1.0, REGION_TINTS[2]]]

show_models = ["逻辑回归", "线性 SVM", "高斯核 SVM (RBF)", "K 近邻 (k=5)"]

# 先在二维投影上重训四种模型，拿到预测区域与测试集准确率
regions, accs = {}, {}
for name in show_models:
    model = clone(models[name]).fit(X_pca, y_train)
    regions[name] = model.predict(grid).reshape(xx.shape)
    accs[name] = accuracy_score(y_test, model.predict(X_test_pca))

fig = make_subplots(rows=1, cols=4, horizontal_spacing=0.04,
                    subplot_titles=[f"{n}<br>二维投影准确率 {accs[n]:.3f}" for n in show_models])

for k, name in enumerate(show_models, start=1):
    region = regions[name]
    fig.add_trace(go.Heatmap(
        x=xx[0], y=yy[:, 0], z=region, colorscale=region_scale, showscale=False,
        hoverinfo="skip", zsmooth=False,
    ), row=1, col=k)
    for cls, (cname, color) in enumerate(zip(wine.target_names, CLASS_COLORS)):
        m = y_train == cls
        fig.add_trace(go.Scatter(
            x=X_pca[m, 0], y=X_pca[m, 1], mode="markers", name=cname, legendgroup=cname,
            showlegend=(k == 1), marker=dict(size=7, color=color, line=dict(width=0.8, color="#fcfcfb")),
            hovertemplate=f"{cname}<br>PC1 %{{x:.2f}}  PC2 %{{y:.2f}}<extra></extra>",
        ), row=1, col=k)

fig.update_xaxes(title_text="主成分 1", row=1, col=1)
fig.update_yaxes(title_text="主成分 2", row=1, col=1)
style(fig, "不同算法的决策边界（PCA 二维投影，浅色区域为预测类别）", "", "",
      width=1180, height=460)
fig.update_layout(margin=dict(l=70, r=30, t=115, b=80))
fig.show()

两个主成分解释的方差比例： [0.357 0.192] （合计 54.9%）
投影后 PC1 范围 [-4.35, 3.84]，PC2 范围 [-3.81, 3.55]
决策平面网格：320 × 320 = 102,400 个点


可以观察到：

* **逻辑回归 / 线性 SVM** 是直线型边界，把平面切成三块凸区域，形态最简单；
* **高斯核 SVM** 的边界在左上角、右下角变成光滑曲线，能顺着样本分布“兜”住边角，
  体现核方法把数据映射到高维后线性可分的思路；
* **K 近邻** 的边界由若干小片拼接、拐角生硬，是“局部投票”的直接结果——这也是它对噪声样本敏感、
  在高维下容易退化的原因。

需要说明的是，本数据集的三个类别本来就**近似线性可分**，所以四种模型的边界大部分区域彼此重合，
差异主要出现在样本稀疏的边角处；如果换成两类互相嵌套的数据（例如同心圆），
这些模型的边界形态才会出现肉眼可见的巨大差别。
此外二维投影只保留了约 55% 的方差，因此这里的准确率略低于 13 维的结果——
真正决定性能的是**特征维度、样本量与模型归纳偏好的匹配程度**。

## 五、预处理与超参数的影响

### 1. 标准化是否必要

In [13]:
scaled_models = {
    "逻辑回归": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEED)),
    "线性 SVM": make_pipeline(StandardScaler(), SVC(kernel="linear", C=1.0, random_state=SEED)),
    "高斯核 SVM (RBF)": make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale", random_state=SEED)),
    "K 近邻 (k=5)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
}
raw_models = {
    "逻辑回归": LogisticRegression(max_iter=1000, random_state=SEED),
    "线性 SVM": SVC(kernel="linear", C=1.0, random_state=SEED),
    "高斯核 SVM (RBF)": SVC(kernel="rbf", C=1.0, gamma="scale", random_state=SEED),
    "K 近邻 (k=5)": KNeighborsClassifier(n_neighbors=5),
}

scale_rows, warnings_seen = [], 0
for name in scaled_models:
    # 未标准化时逻辑回归会因梯度尺度悬殊而不收敛，这里把警告收集起来计数，避免刷屏
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        acc_raw = cross_val_score(raw_models[name], X_train, y_train, cv=cv, scoring="accuracy").mean()
    warnings_seen += sum(1 for w in caught if issubclass(w.category, ConvergenceWarning))
    acc_std = cross_val_score(scaled_models[name], X_train, y_train, cv=cv, scoring="accuracy").mean()
    scale_rows.append({"模型": name, "未标准化": round(acc_raw, 4), "标准化后": round(acc_std, 4),
                       "提升": round(acc_std - acc_raw, 4)})
print(f"未标准化时共触发 {warnings_seen} 次不收敛警告（ConvergenceWarning），"
      f"标准化后不再出现——这也是“预处理属于建模流程一部分”的直接证据。")
df_scale = pd.DataFrame(scale_rows).set_index("模型").sort_values("提升", ascending=False)
df_scale

未标准化时共触发 5 次不收敛警告（ConvergenceWarning），标准化后不再出现——这也是“预处理属于建模流程一部分”的直接证据。


,未标准化,标准化后,提升
模型,,,
高斯核 SVM (RBF),0.6697,0.992,0.3223
K 近邻 (k=5),0.7177,0.968,0.2503
逻辑回归,0.9357,0.984,0.0483
线性 SVM,0.9600,0.984,0.0240


In [14]:
fig = go.Figure()
fig.add_trace(go.Bar(
    x=df_scale.index, y=df_scale["未标准化"], name="未标准化", marker_color=C_ORANGE,
    text=df_scale["未标准化"], textposition="outside", textfont=dict(color=C_MUTED, size=11),
    hovertemplate="%{x}<br>未标准化 %{y:.4f}<extra></extra>",
))
fig.add_trace(go.Bar(
    x=df_scale.index, y=df_scale["标准化后"], name="标准化后", marker_color=C_BLUE,
    text=df_scale["标准化后"], textposition="outside", textfont=dict(color=C_MUTED, size=11),
    hovertemplate="%{x}<br>标准化后 %{y:.4f}<extra></extra>",
))
fig.update_layout(barmode="group", bargap=0.35)
fig.update_yaxes(range=[0, 1.1], dtick=0.2, title_text="5 折 CV 准确率")
style(fig, "标准化对依赖距离/梯度的模型影响显著", "", "5 折 CV 准确率", width=860, height=470)
fig.show()

### 2. 超参数的影响

In [15]:
# K 近邻的 k：k 太小易受噪声干扰，太大则把邻域内其他类别也拉进来
# 标准化放进 pipeline，保证交叉验证每一折里的均值方差都只用该折的训练部分估计
ks = list(range(1, 26))
knn_acc = [cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)),
                           X_train, y_train, cv=cv, scoring="accuracy").mean() for k in ks]

# 高斯核 SVM 的惩罚系数 C：C 越大越迁就训练样本，越小正则化越强
Cs = np.logspace(-2, 3, 20)
svm_acc = [cross_val_score(make_pipeline(StandardScaler(),
                                         SVC(kernel="rbf", C=C, gamma="scale", random_state=SEED)),
                           X_train, y_train, cv=cv, scoring="accuracy").mean() for C in Cs]

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                    subplot_titles=("K 近邻：邻居数 k 的影响", "高斯核 SVM：惩罚系数 C 的影响（对数横轴）"))
fig.add_trace(go.Scatter(
    x=ks, y=knn_acc, mode="lines+markers", name="KNN", showlegend=False,
    line=dict(color=C_BLUE, width=2), marker=dict(size=7),
    hovertemplate="k = %{x}<br>CV 准确率 %{y:.4f}<extra></extra>",
), row=1, col=1)
best_k = ks[int(np.argmax(knn_acc))]
fig.add_annotation(x=best_k, y=max(knn_acc), text=f"最优 k = {best_k}", showarrow=True,
                   arrowhead=0, ax=40, ay=-30, font=dict(color=C_MUTED, size=11),
                   bgcolor="rgba(252,252,251,0.9)", bordercolor="#d8d7d2", borderwidth=1, row=1, col=1)

fig.add_trace(go.Scatter(
    x=Cs, y=svm_acc, mode="lines+markers", name="RBF-SVM", showlegend=False,
    line=dict(color=C_BLUE, width=2), marker=dict(size=7),
    hovertemplate="C = %{x:.3g}<br>CV 准确率 %{y:.4f}<extra></extra>",
), row=1, col=2)
best_C = Cs[int(np.argmax(svm_acc))]
fig.add_annotation(x=best_C, y=max(svm_acc), text=f"最优 C = {best_C:.3g}", showarrow=True,
                   arrowhead=0, ax=-50, ay=-30, font=dict(color=C_MUTED, size=11),
                   bgcolor="rgba(252,252,251,0.9)", bordercolor="#d8d7d2", borderwidth=1, row=1, col=2)

fig.update_xaxes(title_text="邻居数 k", row=1, col=1)
fig.update_xaxes(title_text="惩罚系数 C", type="log", row=1, col=2)
fig.update_yaxes(title_text="5 折 CV 准确率", row=1, col=1)
style(fig, "超参数对模型性能的影响（纵轴为训练集上的 5 折 CV 准确率）", "", "",
      width=1080, height=450)
fig.show()

In [16]:
# 随机森林的特征重要性：与前面 F 检验的排序对照
rf = RandomForestClassifier(n_estimators=300, random_state=SEED).fit(X_train, y_train)
imp = pd.Series(rf.feature_importances_, index=wine.feature_names).sort_values()

fig = go.Figure(go.Bar(
    x=imp.values, y=imp.index, orientation="h", marker_color=C_BLUE,
    hovertemplate="%{y}<br>重要性 %{x:.3f}<extra></extra>",
))
style(fig, "随机森林的特征重要性排序", "重要性（不纯度下降占比）", "", width=760, height=520)
fig.show()